# ✍️ Signature Forgery Detector — **Graph Neural Network Edition**
# Skeleton Graphs + Siamese GNN (Colab + Drive + Disk Cache)

## What changed vs the EfficientNet notebook
| Stage | Before (CNN) | After (GNN) |
|---|---|---|
| Representation | Raw RGB pixels → EfficientNet-B0 CNN embedding | Grayscale → adaptive threshold → **skeleton** → **structural graph** → GNN embedding |
| What the model compares | Texture/appearance of the whole image | **Geometric topology**: stroke endpoints, junctions, turning points, their connectivity, orientation, curvature |
| Backbone | `efficientnet_b0` (ImageNet pretrained) | `GATv2Conv` Graph Attention Encoder (trained from scratch on graphs) |
| Cache | LMDB of resized RGB arrays | Disk cache of serialized `torch_geometric.data.Data` graphs |
| Loss | Contrastive Loss on image embeddings | **Same** Contrastive Loss, now on graph embeddings |
| Verification pipeline | Pairs, threshold sweep, FAR/FRR, ROC, F1, per-writer acc | **Unchanged** — kept end-to-end |

### Why a graph instead of an image?
A signature is fundamentally a *stroke trajectory*, not a texture. Two genuine signatures from the
same writer can differ in ink thickness, scan brightness or stray pen pressure (things a CNN is very
sensitive to), while still sharing the exact same **skeletal structure**: the same loops, the same
crossing points, the same number of endpoints, the same overall stroke geometry. By skeletonizing the
signature down to a 1‑pixel‑wide stroke and converting it into a graph of {endpoints, junctions,
turning points} connected by stroke segments, we force the model to learn writer-specific *structure*
rather than incidental pixel appearance — which is exactly what forensic document examiners look at.

### Pipeline
```
Image
  → preprocessing (autocrop + grayscale)
  → adaptive thresholding (binarize ink)
  → skeletonization (skimage, 1-px-wide skeleton)
  → graph extraction (sknw: endpoints / junctions / turning points as nodes,
                       skeleton paths as edges, with x, y, orientation, degree, curvature features)
  → Siamese Graph Neural Network (GATv2 encoder)
  → graph embedding (L2-normalized, 256-d)
  → Contrastive Loss on Euclidean distance between embeddings
  → verification (threshold sweep, Accuracy, F1, AUC, FAR, FRR, ROC, confusion matrix)
```

**Everything from data loading / writer indexing / writer-aware pairing strategy / evaluation
suite of the original notebook is preserved.** Only the representation (cells under sections 6–9)
is replaced — section numbers below mirror the original notebook so it's easy to diff.

In [ ]:
# Core deps (same as before) + graph / GNN stack
# - scikit-image : adaptive thresholding & skeletonization
# - networkx     : graph data structure
# - sknw         : robust, well-tested conversion of a binary skeleton image into a
#                  NetworkX graph (handles junction merging, spur pruning, pixel-path
#                  bookkeeping far more robustly than a hand-rolled corner/junction detector)
# - torch_geometric : GNN layers (GATv2Conv), graph batching, pooling
!pip install torch torchvision matplotlib scikit-learn pillow numpy pandas seaborn tqdm lmdb -q
!pip install scikit-image networkx sknw -q
!pip install torch_geometric -q

In [ ]:
import os, re, random, json, zipfile, shutil, time, hashlib, pickle
from collections import defaultdict
from datetime import datetime
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# --- graph extraction stack ---
import cv2
import networkx as nx
import sknw
from skimage.morphology import skeletonize

# --- GNN stack ---
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GATv2Conv, global_mean_pool, global_max_pool

from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    classification_report, roc_auc_score, roc_curve, f1_score
)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🖥  Device : {DEVICE}')
print(f'📦 PyTorch: {torch.__version__}')
if torch.cuda.is_available():
    print(f'🎮 GPU    : {torch.cuda.get_device_name(0)}')
    print(f'   VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  GPU tidak terdeteksi — pastikan Runtime -> Change runtime type -> GPU')

## 2. 🔗 Mount Google Drive & Lokasi ZIP

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ZIP_PATH = '/content/drive/MyDrive/gpds.zip'

if not os.path.exists(DRIVE_ZIP_PATH):
    print(f'❌ Tidak ditemukan: {DRIVE_ZIP_PATH}')
    print('📤 Upload archive.zip manual sekarang...')
    from google.colab import files
    uploaded = files.upload()
    local_name = list(uploaded.keys())[0]
    os.makedirs(os.path.dirname(DRIVE_ZIP_PATH), exist_ok=True)
    shutil.copy(local_name, DRIVE_ZIP_PATH)
    print(f'✅ Disimpan ke Drive: {DRIVE_ZIP_PATH}')
else:
    print(f'✅ ZIP ditemukan di Drive: {DRIVE_ZIP_PATH}')
    print(f'   Size: {os.path.getsize(DRIVE_ZIP_PATH) / 1e6:.1f} MB')

## 3. 📂 Extract ZIP ke Disk Lokal Colab

In [ ]:
ZIP_PATH  = DRIVE_ZIP_PATH
DATA_DIR  = '/content/data/custom'
ZIP_INNER = 'New folder (10)/test'
EXTS      = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')

if not os.path.exists(DATA_DIR) or len(os.listdir(DATA_DIR)) == 0:
    print(f'📦 Extracting {ZIP_PATH} -> {DATA_DIR} ...')
    os.makedirs(DATA_DIR, exist_ok=True)
    t0 = time.time()
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        members = [m for m in zf.namelist() if ZIP_INNER in m and not m.endswith('/')]
        for member in tqdm(members, desc='Extracting'):
            parts = member.replace('\\', '/').split('/')
            try:
                idx = parts.index('test')
                rel = parts[idx + 1:]
            except ValueError:
                continue
            if len(rel) < 3:
                continue
            dest = os.path.join(DATA_DIR, *rel)
            os.makedirs(os.path.dirname(dest), exist_ok=True)
            with zf.open(member) as src, open(dest, 'wb') as dst:
                dst.write(src.read())
    print(f'✅ Extracted in {time.time() - t0:.1f}s -> {DATA_DIR}')
else:
    print(f'✅ Dataset sudah ada di {DATA_DIR} (skip extract)')

gen_count  = sum(1 for root, _, files in os.walk(DATA_DIR)
                 for f in files if 'genuine' in root and f.lower().endswith(EXTS))
forg_count = sum(1 for root, _, files in os.walk(DATA_DIR)
                 for f in files if 'forge' in root and f.lower().endswith(EXTS))
print(f'\n📊 Dataset Summary')
print(f'   Genuine : {gen_count}')
print(f'   Forged  : {forg_count}')
print(f'   Total   : {gen_count + forg_count}')

## 4. 🗂 Writer-Aware File Index

In [ ]:
def build_index(data_dir, label_folder):
    index = defaultdict(list)
    for writer_id in sorted(os.listdir(data_dir)):
        folder = os.path.join(data_dir, writer_id, label_folder)
        if not os.path.isdir(folder):
            continue
        for fname in sorted(os.listdir(folder)):
            if fname.lower().endswith(EXTS):
                index[writer_id].append(os.path.join(folder, fname))
    return dict(index)

genuine_index = build_index(DATA_DIR, 'genuine')
forged_index  = build_index(DATA_DIR, 'forge')

writers = sorted(set(genuine_index) & set(forged_index))
print(f'Writers with genuine & forged: {len(writers)}')
for wid in writers[:5]:
    print(f'  Writer {wid:>4}: {len(genuine_index[wid])} genuine, {len(forged_index[wid])} forged')

## 5. 🦴 Preprocessing → Skeletonization → Graph Extraction

This replaces the old "resize into an LMDB of RGB arrays" cache with a **graph cache**: every
signature image is converted **once** into a `torch_geometric.data.Data` graph and pickled to
disk, so training never re-runs OpenCV/skeletonization per epoch.

### 5.1 Preprocessing
We keep the original autocrop-to-ink-bbox + square-pad logic (it's still the right fix for huge
white scan margins), but stop before normalization since we need a clean grayscale binary mask,
not an ImageNet-normalized tensor.

### 5.2 Adaptive thresholding
`cv2.adaptiveThreshold` (Gaussian, per-region threshold) is used instead of a single global
threshold because signature scans have uneven illumination; a global threshold either eats faint
pen strokes or leaves scanner noise as spurious ink.

### 5.3 Skeletonization
`skimage.morphology.skeletonize` (Zhang–Suen style thinning) reduces the binarized ink mask to a
**1-pixel-wide** skeleton — the topological backbone of every stroke.

### 5.4 Graph extraction — robust approach
Hand-rolled corner/junction detectors on raw skeleton pixels are notoriously unstable (a single
noisy pixel can create or destroy a junction). Instead we use **`sknw`**, a library purpose-built
and widely used for exactly this: it converts a binary skeleton into a `networkx` graph where
- **nodes** = pixels with degree ≠ 2 in the skeleton graph (i.e. **endpoints**, degree 1, and
  **junctions**, degree ≥ 3), which sknw automatically detects and de-duplicates/merges,
- **edges** = the actual pixel-path of each stroke segment connecting two nodes (`pts` array),
  so we get the *full path geometry*, not just a straight line between nodes.

"Important turning points" (high-curvature points along a long straight stroke that isn't an
endpoint or junction) are added afterwards by walking each edge's pixel path and inserting an
extra node wherever the local turning angle exceeds a threshold — this is the standard
Douglas-Peucker-style approach to keeping graphs robust (curvature-based simplification) instead
of a brittle naive corner detector.

### 5.5 Per-node features
For every node we store: **x, y** (normalized to [0,1] by image size), **degree** (number of
incident strokes — discriminates endpoints/junctions/turning points), **local orientation**
(mean angle, in `[sin, cos]` form, of the incident stroke directions), and **curvature**
(mean absolute turning angle along the incident stroke paths). Edge features store the
**segment length** (normalized) and the **same orientation/curvature** measured along the path.

In [ ]:
IMG_SIZE_CACHE  = 224          # autocrop+pad target before graph extraction
GRAPH_CACHE_DIR = '/content/cache/signature_graphs_v1'
os.makedirs(GRAPH_CACHE_DIR, exist_ok=True)

all_paths = sorted(set(
    p for paths in genuine_index.values() for p in paths
) | set(
    p for paths in forged_index.values()  for p in paths
))
print(f'Total gambar unik untuk diproses jadi graph: {len(all_paths)}')


# ── Preprocessing: autocrop to ink bbox + square pad (kept from original) ──
def autocrop_and_pad(img, pad_frac=0.08, ink_thresh=245, size=IMG_SIZE_CACHE):
    gray = np.array(img.convert('L'))
    mask = gray < ink_thresh
    if mask.sum() == 0:
        cropped = img.convert('L')
    else:
        ys, xs = np.where(mask)
        y0, y1 = int(ys.min()), int(ys.max())
        x0, x1 = int(xs.min()), int(xs.max())
        h, w   = gray.shape
        pad_y  = max(2, int((y1 - y0 + 1) * pad_frac))
        pad_x  = max(2, int((x1 - x0 + 1) * pad_frac))
        y0, y1 = max(0, y0 - pad_y), min(h - 1, y1 + pad_y)
        x0, x1 = max(0, x0 - pad_x), min(w - 1, x1 + pad_x)
        cropped = img.convert('L').crop((x0, y0, x1 + 1, y1 + 1))
    cw, ch = cropped.size
    side   = max(cw, ch)
    canvas = Image.new('L', (side, side), 255)
    canvas.paste(cropped, ((side - cw) // 2, (side - ch) // 2))
    return canvas.resize((size, size), Image.BILINEAR)


def _angle_features(p_from, p_to):
    '''Return (sin, cos) of the direction vector p_from -> p_to.'''
    dy, dx = (p_to[0] - p_from[0]), (p_to[1] - p_from[1])
    ang = np.arctan2(dy, dx)
    return np.sin(ang), np.cos(ang)


def _path_curvature(pts):
    '''Mean absolute turning angle (radians) along a polyline of (row, col) points.'''
    if len(pts) < 3:
        return 0.0
    pts = np.asarray(pts, dtype=np.float32)
    v = np.diff(pts, axis=0)
    ang = np.arctan2(v[:, 0], v[:, 1])
    dang = np.diff(ang)
    dang = (dang + np.pi) % (2 * np.pi) - np.pi   # wrap to [-pi, pi]
    return float(np.mean(np.abs(dang))) if len(dang) else 0.0


def _insert_turning_points(G, curvature_thresh=0.35, min_seg=8):
    '''Add 'important turning point' nodes along long edges where the local
    turning angle exceeds curvature_thresh. Robust, simplification-style
    approach: walk the pixel path in fixed steps and only split at genuine
    high-curvature locations (avoids spurious nodes from single noisy pixels).'''
    new_node_id = max(G.nodes) + 1 if len(G.nodes) else 0
    edges_to_process = list(G.edges(keys=True, data=True)) if G.is_multigraph() \
        else [(u, v, 0, d) for u, v, d in G.edges(data=True)]
    for u, v, k, data in edges_to_process:
        pts = data.get('pts')
        if pts is None or len(pts) < min_seg * 2:
            continue
        step = max(min_seg, len(pts) // 10)
        for i in range(step, len(pts) - step, step):
            local = pts[max(0, i - step):i + step]
            if _path_curvature(local) > curvature_thresh:
                node_xy = pts[i]
                G.add_node(new_node_id, o=np.array([node_xy[0], node_xy[1]]), pts=np.array([node_xy]))
                new_node_id += 1
    return G


def build_signature_graph(path, size=IMG_SIZE_CACHE):
    '''Full preprocessing -> threshold -> skeleton -> graph pipeline for one image.
    Returns a torch_geometric.data.Data object (always >= 2 nodes, never empty).'''
    img = Image.open(path)
    img = autocrop_and_pad(img, size=size)
    arr = np.array(img, dtype=np.uint8)

    # adaptive threshold: ink = white(255) on black(0) for skeletonize
    binary = cv2.adaptiveThreshold(
        arr, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV,
        blockSize=25, C=10)
    binary = (binary > 0)

    skeleton = skeletonize(binary)

    if skeleton.sum() < 10:
        # Degenerate / blank image fallback: synthetic 2-node graph at image
        # centroid of whatever ink exists, so the pipeline never crashes.
        ys, xs = np.where(binary) if binary.sum() > 0 else (np.array([size//2]), np.array([size//2]))
        cy, cx = ys.mean() / size, xs.mean() / size
        x = torch.tensor([[cx, cy, 0.0, 0.0, 0.0, 0.0],
                           [cx, cy, 0.0, 0.0, 0.0, 0.0]], dtype=torch.float32)
        edge_index = torch.tensor([[0, 1], [1, 0]], dtype=torch.long)
        edge_attr  = torch.zeros((2, 3), dtype=torch.float32)
        return Data(x=x, edge_index=edge_index, edge_attr=edge_attr)

    G = sknw.build_sknw(skeleton, multi=True)
    G = _insert_turning_points(G)

    nodes = list(G.nodes)
    node_idx = {n: i for i, n in enumerate(nodes)}

    # --- per-node aggregation buffers ---
    n = len(nodes)
    incident_dirs = [[] for _ in range(n)]   # list of (sin, cos)
    incident_curv = [[] for _ in range(n)]

    edge_src, edge_dst, edge_feat = [], [], []
    edge_iter = G.edges(keys=True, data=True) if G.is_multigraph() else \
        [(u, v, 0, d) for u, v, d in G.edges(data=True)]
    for u, v, k, data in edge_iter:
        if u not in node_idx or v not in node_idx:
            continue
        iu, iv = node_idx[u], node_idx[v]
        pts = data.get('pts')
        if pts is not None and len(pts) >= 2:
            length = float(np.linalg.norm(pts[-1].astype(np.float32) - pts[0].astype(np.float32))) / size
            s, c = _angle_features(pts[0], pts[-1])
            curv = _path_curvature(pts)
        else:
            ou = G.nodes[u].get('o', np.array([0, 0]))
            ov = G.nodes[v].get('o', np.array([0, 0]))
            length = float(np.linalg.norm(ov - ou)) / size
            s, c = _angle_features(ou, ov)
            curv = 0.0

        feat = [length, s, c]
        for a, b in [(iu, iv), (iv, iu)]:
            edge_src.append(a); edge_dst.append(b)
            edge_feat.append([length, s if a == iu else -s, c])
        incident_dirs[iu].append((s, c)); incident_dirs[iv].append((-s, -c))
        incident_curv[iu].append(curv);   incident_curv[iv].append(curv)

    x_feats = []
    for i, node in enumerate(nodes):
        o = G.nodes[node].get('o', np.array([0, 0]))
        y_coord, x_coord = float(o[0]) / size, float(o[1]) / size
        degree = G.degree(node)
        if incident_dirs[i]:
            s_mean = float(np.mean([d[0] for d in incident_dirs[i]]))
            c_mean = float(np.mean([d[1] for d in incident_dirs[i]]))
            curv_mean = float(np.mean(incident_curv[i]))
        else:
            s_mean = c_mean = curv_mean = 0.0
        x_feats.append([x_coord, y_coord, degree / 8.0, s_mean, c_mean, curv_mean])

    if n < 2:
        # single isolated node: duplicate so we still have a valid edge_index
        x_feats.append(x_feats[0])
        edge_src, edge_dst = [0, 1], [1, 0]
        edge_feat = [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0]]

    x = torch.tensor(x_feats, dtype=torch.float32)
    if len(edge_src) == 0:
        edge_index = torch.empty((2, 0), dtype=torch.long)
        edge_attr  = torch.empty((0, 3), dtype=torch.float32)
    else:
        edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long)
        edge_attr  = torch.tensor(edge_feat, dtype=torch.float32)

    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr)


def _cache_key(path):
    return os.path.join(GRAPH_CACHE_DIR, hashlib.md5(path.encode('utf-8')).hexdigest() + '.pt')


def _build_and_cache(path):
    try:
        ck = _cache_key(path)
        if os.path.exists(ck):
            return path, True
        g = build_signature_graph(path)
        torch.save(g, ck)
        return path, True
    except Exception as e:
        return path, False


print('🦴 Building skeleton graphs (cached to disk, one-time cost)...')
t0 = time.time()
ok, fail = 0, 0
for p in tqdm(all_paths, desc='Graph extraction'):
    _, success = _build_and_cache(p)
    ok += success; fail += (not success)
print(f'✅ Graph cache built: {ok}/{len(all_paths)} ok, {fail} failed, in {time.time()-t0:.1f}s -> {GRAPH_CACHE_DIR}')


def load_graph(path):
    ck = _cache_key(path)
    if not os.path.exists(ck):
        _build_and_cache(path)
    return torch.load(ck, weights_only=False)


# quick sanity check / visualization of one extracted graph
sample_path = all_paths[0]
g = load_graph(sample_path)
print(f'\n🔍 Sample graph for {os.path.basename(sample_path)}:')
print(f'   nodes={g.num_nodes}  edges={g.num_edges}  node_feat_dim={g.num_node_features}  edge_feat_dim={g.num_edge_features}')

## 6. 🔗 Writer-Dependent Pair Dataset (Graph-backed)

Identical writer-anchored pairing strategy and pair ratio (50% positive / 25% same-writer
genuine-vs-forged / 25% different-writer impostor) as the original notebook — only `_load`
changed, from "decode an RGB array out of LMDB" to "load a cached `Data` graph object", and the
collate step now batches **graphs** (via `torch_geometric.data.Batch`) instead of stacking image
tensors.

In [ ]:
class WriterDependentGraphDataset(Dataset):
    '''
    Generates writer-anchored Siamese GRAPH pairs.
      label=0 : same writer, both genuine (positive)
      label=1 : same writer genuine vs forged (hard negative)
               OR different writer genuine (impostor negative)

    Pair ratio — 40% pos / 35% neg_forged / 25% neg_impostor (unchanged from original).
    Pairs are freshly resampled every epoch via .resample(epoch) (unchanged from original) so
    the model never memorizes a fixed pair list.
    '''
    def __init__(self, genuine_index, forged_index, writers, augment=False,
                 n_pairs=3000, seed=SEED):
        self.genuine_index = genuine_index
        self.forged_index  = forged_index
        self.writers       = writers
        self.n_pairs        = n_pairs
        self.base_seed      = seed
        self.augment        = augment
        self.epoch           = 0
        self.pairs, self.labels = self._draw_pairs(seed)

    def _draw_pairs(self, seed):
        rng = np.random.RandomState(seed)
        n_pos          = int(self.n_pairs * 0.40)
        n_neg_forged   = int(self.n_pairs * 0.35)
        n_neg_impostor = self.n_pairs - n_pos - n_neg_forged

        pairs, labels = [], []
        for _ in range(n_pos):
            wid    = rng.choice(self.writers)
            g      = self.genuine_index[wid]
            p1, p2 = (rng.choice(g, 2, replace=False) if len(g) >= 2 else (g[0], g[0]))
            pairs.append((p1, p2)); labels.append(0)
        for _ in range(n_neg_forged):
            wid = rng.choice(self.writers)
            pairs.append((rng.choice(self.genuine_index[wid]),
                          rng.choice(self.forged_index[wid])))
            labels.append(1)
        for _ in range(n_neg_impostor):
            wa, wb = rng.choice(self.writers, 2, replace=False)
            pairs.append((rng.choice(self.genuine_index[wa]),
                          rng.choice(self.genuine_index[wb])))
            labels.append(1)

        combined = list(zip(pairs, labels))
        rng.shuffle(combined)
        pairs, labels = zip(*combined)
        return list(pairs), list(labels)

    def resample(self, epoch):
        self.epoch = epoch
        self.pairs, self.labels = self._draw_pairs(self.base_seed + epoch)

    def _augment_graph(self, data):
        '''Light structural augmentation: small jitter on node coordinates +
        random edge dropout, analogous to the CNN's RandomAffine/RandomErasing.'''
        data = data.clone()
        if data.num_nodes > 0:
            jitter = torch.randn_like(data.x[:, :2]) * 0.01
            data.x[:, :2] = (data.x[:, :2] + jitter).clamp(0, 1)
        if self.augment and data.edge_index.size(1) > 4 and random.random() < 0.3:
            keep = torch.rand(data.edge_index.size(1)) > 0.1
            if keep.sum() >= 2:
                data.edge_index = data.edge_index[:, keep]
                data.edge_attr  = data.edge_attr[keep]
        return data

    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        p1, p2 = self.pairs[idx]
        label  = self.labels[idx]
        g1 = self._augment_graph(load_graph(p1)) if self.augment else load_graph(p1)
        g2 = self._augment_graph(load_graph(p2)) if self.augment else load_graph(p2)
        return g1, g2, torch.tensor(label, dtype=torch.float32)


def graph_pair_collate(batch):
    g1_list, g2_list, labels = zip(*batch)
    return Batch.from_data_list(g1_list), Batch.from_data_list(g2_list), torch.stack(labels)


rng_split = np.random.RandomState(SEED)
shuffled  = list(writers); rng_split.shuffle(shuffled)
cut       = int(len(shuffled) * 0.8)
train_writers, val_writers = shuffled[:cut], shuffled[cut:]
print(f'Train writers: {len(train_writers)} | Val writers: {len(val_writers)}')

total_imgs    = sum(len(v) for v in genuine_index.values())
n_train_pairs = max(3000, min(9000, total_imgs * 5))
n_val_pairs   = max(800,  min(2000, total_imgs))

train_dataset = WriterDependentGraphDataset(
    genuine_index, forged_index, train_writers, augment=True,  n_pairs=n_train_pairs)
val_dataset = WriterDependentGraphDataset(
    genuine_index, forged_index, val_writers,   augment=False, n_pairs=n_val_pairs)

NUM_WORKERS = min(4, os.cpu_count() or 2)   # graphs are CPU-light once cached, no need for 8 workers
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,
                          num_workers=NUM_WORKERS, collate_fn=graph_pair_collate)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False,
                          num_workers=NUM_WORKERS, collate_fn=graph_pair_collate)

print(f'\n✅ Train: {len(train_dataset)} pairs | Val: {len(val_dataset)} pairs')
print(f'   Pair ratio — pos:neg_forged:neg_impostor = 40%:35%:25%  (kept from original)')

## 7. 🧠 Siamese Graph Neural Network Encoder

`GATv2Conv` (graph attention) layers were chosen over a plain GCN/GraphSAGE because signature
graphs are small and irregular — attention lets each node (an endpoint, junction or turning
point) learn *which* of its neighboring strokes matter most for distinguishing a writer's style,
rather than averaging all neighbors uniformly. Three stacked GAT layers give each node a
receptive field covering most of a typical signature's stroke graph.

After message passing, node embeddings are pooled into one **graph-level embedding** using both
mean and max global pooling concatenated together (mean captures overall structure, max captures
the most salient distinguishing stroke/junction), then projected through the same
`Linear → BatchNorm → ReLU → Dropout → Linear` head as the original CNN embedder, and L2
normalized — so the rest of the pipeline (contrastive loss, distance threshold, metrics) is a
drop-in match for the old embedder.

In [ ]:
class GraphEmbedder(nn.Module):
    def __init__(self, node_feat_dim=6, edge_feat_dim=3, hidden_dim=128,
                 embedding_dim=256, heads=4, dropout=0.3):
        super().__init__()
        self.conv1 = GATv2Conv(node_feat_dim, hidden_dim, heads=heads,
                               edge_dim=edge_feat_dim, dropout=dropout, concat=True)
        self.conv2 = GATv2Conv(hidden_dim * heads, hidden_dim, heads=heads,
                               edge_dim=edge_feat_dim, dropout=dropout, concat=True)
        self.conv3 = GATv2Conv(hidden_dim * heads, hidden_dim, heads=1,
                               edge_dim=edge_feat_dim, dropout=dropout, concat=False)
        self.bn1 = nn.BatchNorm1d(hidden_dim * heads)
        self.bn2 = nn.BatchNorm1d(hidden_dim * heads)

        pooled_dim = hidden_dim * 2   # mean-pool + max-pool concatenated
        self.head = nn.Sequential(
            nn.Linear(pooled_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, embedding_dim)
        )

    def forward(self, data):
        x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch
        x = F.elu(self.bn1(self.conv1(x, edge_index, edge_attr)))
        x = F.elu(self.bn2(self.conv2(x, edge_index, edge_attr)))
        x = F.elu(self.conv3(x, edge_index, edge_attr))

        x_mean = global_mean_pool(x, batch)
        x_max  = global_max_pool(x, batch)
        x = torch.cat([x_mean, x_max], dim=1)

        x = self.head(x)
        return F.normalize(x, p=2, dim=1)


class ContrastiveLoss(nn.Module):
    '''Unchanged from the original notebook.'''
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin

    def forward(self, emb1, emb2, label):
        dist = F.pairwise_distance(emb1, emb2)
        loss = (1 - label) * dist.pow(2) + label * F.relu(self.margin - dist).pow(2)
        return loss.mean(), dist


model     = GraphEmbedder(node_feat_dim=6, edge_feat_dim=3,
                          hidden_dim=128, embedding_dim=256, heads=4, dropout=0.3).to(DEVICE)
criterion = ContrastiveLoss(margin=1.0)

total_p     = sum(p.numel() for p in model.parameters())
trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\n✅ Model ready (Siamese GNN — GATv2 encoder)')
print(f'   Total parameters     : {total_p:,}')
print(f'   Trainable parameters : {trainable_p:,}')

## 8. 🏋 Training — Two-Stage LR Schedule

The original "Phase 1 frozen backbone / Phase 2 unfreeze last blocks" split existed because the
CNN backbone started from **ImageNet-pretrained weights** that needed protecting from early
high-LR gradient noise. The GNN here is trained **from scratch** on graphs (there is no
pretrained graph backbone to protect), so there is nothing to "freeze" — but we keep the same
**two-stage structure** for comparability and because it still helps: Stage 1 trains the full
network with a higher LR to learn coarse structural separability quickly; Stage 2 continues
training at 10× lower LR with a fresh cosine schedule to fine-tune the embedding geometry,
mirroring the original notebook's warm-up → fine-tune pattern.

In [ ]:
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE.type == 'cuda'))

def to_device(batch_graph, device):
    return batch_graph.to(device, non_blocking=True)

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for g1, g2, labels in loader:
        g1, g2 = to_device(g1, device), to_device(g2, device)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            emb1 = model(g1); emb2 = model(g2)
            loss, dists = criterion(emb1, emb2, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
        correct    += ((dists > 0.5).float() == labels).sum().item()
        total      += labels.size(0)
    return total_loss / len(loader), correct / total

def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_labels, all_dists = [], []
    with torch.no_grad():
        for g1, g2, labels in loader:
            g1, g2 = to_device(g1, device), to_device(g2, device)
            labels = labels.to(device, non_blocking=True)
            with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                emb1 = model(g1); emb2 = model(g2)
                loss, dists = criterion(emb1, emb2, labels)
            total_loss += loss.item()
            correct    += ((dists > 0.5).float() == labels).sum().item()
            total      += labels.size(0)
            all_labels.extend(labels.cpu().numpy())
            all_dists.extend(dists.float().cpu().numpy())
    return total_loss / len(loader), correct / total, all_labels, all_dists


PHASE1_EPOCHS = 15
PHASE2_EPOCHS = 15
TOTAL_EPOCHS  = PHASE1_EPOCHS + PHASE2_EPOCHS

history      = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'phase': []}
best_val_acc = 0.0

optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=PHASE1_EPOCHS)

print('=' * 70)
print(f'STAGE 1 -- Warm-up ({PHASE1_EPOCHS} epochs, lr=1e-3)')
print('=' * 70)
print(f'{"Epoch":>6} | {"Train Loss":>10} | {"Train Acc":>9} | {"Val Loss":>8} | {"Val Acc":>7} | {"sec/epoch":>9}')
print('-' * 70)

for epoch in range(1, PHASE1_EPOCHS + 1):
    t0 = time.time()
    train_dataset.resample(epoch)
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion, DEVICE)
    va_loss, va_acc, all_labels, all_dists = eval_epoch(model, val_loader, criterion, DEVICE)
    scheduler.step()
    history['train_loss'].append(tr_loss); history['val_loss'].append(va_loss)
    history['train_acc'].append(tr_acc);   history['val_acc'].append(va_acc)
    history['phase'].append(1)
    flag = ''
    if va_acc > best_val_acc:
        best_val_acc = va_acc
        torch.save(model.state_dict(), '/content/best_gnn_signature.pth')
        flag = ' <- best'
    dt = time.time() - t0
    print(f'{epoch:>6} | {tr_loss:>10.4f} | {tr_acc:>8.2%} | {va_loss:>8.4f} | {va_acc:>6.2%} | {dt:>8.1f}s{flag}')

print(f'\n✅ Stage 1 complete. Best so far: {best_val_acc:.2%}')

In [ ]:
# Stage 2 -- continue training at lower LR (fine-tune embedding geometry)
optimizer2 = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler2 = optim.lr_scheduler.CosineAnnealingLR(optimizer2, T_max=PHASE2_EPOCHS)

print()
print('=' * 70)
print(f'STAGE 2 -- Fine-tune ({PHASE2_EPOCHS} epochs, lr=1e-4)')
print('=' * 70)
print(f'{"Epoch":>6} | {"Train Loss":>10} | {"Train Acc":>9} | {"Val Loss":>8} | {"Val Acc":>7} | {"sec/epoch":>9}')
print('-' * 70)

for epoch in range(1, PHASE2_EPOCHS + 1):
    t0 = time.time()
    train_dataset.resample(PHASE1_EPOCHS + epoch)
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer2, criterion, DEVICE)
    va_loss, va_acc, all_labels, all_dists = eval_epoch(model, val_loader, criterion, DEVICE)
    scheduler2.step()
    history['train_loss'].append(tr_loss); history['val_loss'].append(va_loss)
    history['train_acc'].append(tr_acc);   history['val_acc'].append(va_acc)
    history['phase'].append(2)
    flag = ''
    if va_acc > best_val_acc:
        best_val_acc = va_acc
        torch.save(model.state_dict(), '/content/best_gnn_signature.pth')
        flag = ' <- best'
    dt = time.time() - t0
    print(f'{epoch:>6} | {tr_loss:>10.4f} | {tr_acc:>8.2%} | {va_loss:>8.4f} | {va_acc:>6.2%} | {dt:>8.1f}s{flag}')

print(f'\n✅ Training complete! Best Val Accuracy: {best_val_acc:.2%}')

## 9. 📊 Evaluate Performance

Same evaluation suite as the original notebook: macro-F1 threshold sweep, classification report,
confusion matrix, ROC/AUC, and FAR/FRR — now computed on graph-embedding distances instead of
image-embedding distances.

In [ ]:
model.load_state_dict(torch.load('/content/best_gnn_signature.pth', map_location=DEVICE))
_, _, all_labels, all_dists = eval_epoch(model, val_loader, criterion, DEVICE)

thresholds = np.arange(0.05, 1.51, 0.01)
f1_scores  = [f1_score(all_labels,
                       [1 if d > t else 0 for d in all_dists],
                       average='macro')
              for t in thresholds]
THRESHOLD  = float(thresholds[np.argmax(f1_scores)])

print(f'🔍 Threshold sweep result:')
print(f'   Optimal threshold : {THRESHOLD:.2f}  (macro-F1 = {max(f1_scores):.4f})')

In [ ]:
all_preds = [1 if d > THRESHOLD else 0 for d in all_dists]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
ep             = list(range(1, TOTAL_EPOCHS + 1))
phase_boundary = PHASE1_EPOCHS + 0.5

for ax, metric, ylabel in zip(
    axes,
    [('train_loss', 'val_loss'), ('train_acc', 'val_acc')],
    ['Contrastive Loss', 'Accuracy (%)']
):
    y1 = history[metric[0]]; y2 = history[metric[1]]
    if ylabel == 'Accuracy (%)':
        y1 = [v*100 for v in y1]; y2 = [v*100 for v in y2]
    ax.plot(ep, y1, label='Train', color='#378ADD', lw=2)
    ax.plot(ep, y2, label='Val',   color='#D85A30', lw=2)
    ax.axvline(phase_boundary, color='gray', ls='--', lw=1.2, alpha=0.7, label='Stage 1->2')
    ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
print('=' * 50)
print('📋 CLASSIFICATION REPORT')
print('=' * 50)
print(classification_report(all_labels, all_preds, target_names=['Match ✅', 'No Match ❌']))

cm          = confusion_matrix(all_labels, all_preds)
fpr, tpr, _ = roc_curve(all_labels, all_dists)
auc_score   = roc_auc_score(all_labels, all_dists)

# FAR / FRR at the chosen operating threshold
tn, fp, fn, tp = cm.ravel()
FAR = fp / (fp + tn) if (fp + tn) > 0 else 0.0   # impostors wrongly accepted
FRR = fn / (fn + tp) if (fn + tp) > 0 else 0.0   # genuine wrongly rejected
print(f'FAR (False Accept Rate) : {FAR:.4f}')
print(f'FRR (False Reject Rate) : {FRR:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Match', 'No Match'],
            yticklabels=['Match', 'No Match'], ax=axes[0])
axes[0].set_title('Confusion Matrix', fontsize=12)
axes[0].set_ylabel('Actual'); axes[0].set_xlabel('Predicted')
axes[1].plot(fpr, tpr, color='#378ADD', lw=2, label=f'ROC AUC = {auc_score:.3f}')
axes[1].plot([0,1],[0,1], 'k--', lw=1)
axes[1].set_title('ROC Curve', fontsize=12)
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR'); axes[1].legend()
plt.tight_layout(); plt.show()

acc = accuracy_score(all_labels, all_preds)
print(f'\n✅ Accuracy : {acc:.2%}')
print(f'✅ F1 (macro): {max(f1_scores):.4f}')
print(f'✅ AUC      : {auc_score:.4f}')

In [ ]:
print('\n📊 Per-Writer Verification Accuracy (val writers)')
per_writer_acc = {}

model.eval()
with torch.no_grad():
    for wid in val_writers[:10]:
        g_paths = genuine_index.get(wid, [])
        f_paths = forged_index.get(wid, [])
        if len(g_paths) < 2 or len(f_paths) < 1:
            continue
        correct_w, total_w = 0, 0
        ref_graph = Batch.from_data_list([load_graph(g_paths[0])]).to(DEVICE)
        ref_e     = model(ref_graph)
        for p in g_paths[1:]:
            gb = Batch.from_data_list([load_graph(p)]).to(DEVICE)
            d  = F.pairwise_distance(ref_e, model(gb)).item()
            correct_w += (d < THRESHOLD); total_w += 1
        for p in f_paths:
            gb = Batch.from_data_list([load_graph(p)]).to(DEVICE)
            d  = F.pairwise_distance(ref_e, model(gb)).item()
            correct_w += (d >= THRESHOLD); total_w += 1
        per_writer_acc[wid] = correct_w / total_w if total_w > 0 else 0
        print(f'   Writer {wid:>4}: {per_writer_acc[wid]:.1%} ({correct_w}/{total_w})')

if per_writer_acc:
    print(f'\n   Mean per-writer acc: {np.mean(list(per_writer_acc.values())):.2%}')

## 10. 💾 Save & Export — balik ke Google Drive

In [ ]:
total_params    = sum(p.numel() for p in model.parameters())
trainable_final = sum(p.numel() for p in model.parameters() if p.requires_grad)

LOCAL_MODEL_PATH = '/content/GNN-Siamese-signature.pth'
LOCAL_META_PATH  = '/content/gnn_signature_metadata.json'

torch.save({
    'model_state_dict' : model.state_dict(),
    'best_val_acc'     : best_val_acc,
    'threshold'        : THRESHOLD,
    'embedding_dim'    : 256,
    'node_feat_dim'    : 6,
    'edge_feat_dim'    : 3,
    'val_writers'      : val_writers,
    'dataset'          : 'custom (archive.zip)',
    'saved_at'         : datetime.now().isoformat()
}, LOCAL_MODEL_PATH)

metadata = {
    'project'          : 'Signature Forgery Detector -- Siamese GNN (Graph-of-Skeleton)',
    'representation'   : 'adaptive-threshold -> skeletonize -> sknw graph (endpoints/junctions/turning points)',
    'encoder'          : 'GATv2 x3 + mean/max global pooling + projection head',
    'dataset'          : 'Custom archive.zip',
    'dataset_structure': 'test/{writer_id}/{genuine|forge}/*.jpg',
    'total_writers'    : len(writers),
    'train_writers'    : len(train_writers),
    'val_writers_count': len(val_writers),
    'pairing_strategy' : '40% pos / 35% neg_forged / 25% neg_impostor, resampled every epoch',
    'loss'             : 'Contrastive Loss (margin=1.0) on graph embeddings',
    'data_pipeline'    : 'autocrop-to-ink + square-pad -> adaptive threshold -> skeletonize -> sknw graph, disk-cached as .pt',
    'threshold_method' : f'Optimal via macro-F1 sweep [0.05, 1.50], value={THRESHOLD:.2f}',
    'training_stages'  : [
        {'stage': 1, 'epochs': PHASE1_EPOCHS, 'lr': 1e-3},
        {'stage': 2, 'epochs': PHASE2_EPOCHS, 'lr': 1e-4}
    ],
    'best_val_acc'     : round(best_val_acc, 4),
    'threshold'        : THRESHOLD,
    'embedding_dim'    : 256,
    'total_params'     : total_params,
    'training_history' : history
}
with open(LOCAL_META_PATH, 'w') as f:
    json.dump(metadata, f, indent=2)

DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/signature_dataset/outputs'
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
shutil.copy(LOCAL_MODEL_PATH, DRIVE_OUTPUT_DIR)
shutil.copy(LOCAL_META_PATH,  DRIVE_OUTPUT_DIR)

print('✅ Model   :', os.path.join(DRIVE_OUTPUT_DIR, os.path.basename(LOCAL_MODEL_PATH)))
print('✅ Metadata:', os.path.join(DRIVE_OUTPUT_DIR, os.path.basename(LOCAL_META_PATH)))
print()
print('📊 Final Summary')
print(f'   Dataset              : Custom archive.zip')
print(f'   Total writers        : {len(writers)}')
print(f'   Train / Val writers  : {len(train_writers)} / {len(val_writers)}')
print(f'   Best Val Accuracy    : {best_val_acc:.2%}')
print(f'   Optimal Threshold    : {THRESHOLD:.2f}')
print(f'   Embedding dim        : 256')
print(f'   Representation       : skeleton graph (sknw) -> Siamese GATv2 GNN')

## 🔍 Bonus: Cek GPU Utilization

In [ ]:
!nvidia-smi --query-gpu=utilization.gpu,utilization.memory,memory.used,memory.total --format=csv